In [ ]:
##### 해당 파일은 코랩환경에서 작업했습니다 #####

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# target 준비

In [2]:
from pathlib import Path
import re
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
import glob
import json
import orjson

In [3]:
# 1. 자모 리스트 정의
CHOSUNG_LIST = ['ㄱ','ㄲ','ㄴ','ㄷ','ㄸ','ㄹ','ㅁ','ㅂ','ㅃ','ㅅ','ㅆ',
                'ㅇ','ㅈ','ㅉ','ㅊ','ㅋ','ㅌ','ㅍ','ㅎ']
JUNGSUNG_LIST = ['ㅏ','ㅐ','ㅑ','ㅒ','ㅓ','ㅔ','ㅕ','ㅖ','ㅗ','ㅘ','ㅙ',
                 'ㅚ','ㅛ','ㅜ','ㅝ','ㅞ','ㅟ','ㅠ','ㅡ','ㅢ','ㅣ']
JONGSUNG_LIST = ['','ㄱ','ㄲ','ㄳ','ㄴ','ㄵ','ㄶ','ㄷ','ㄹ','ㄺ','ㄻ',
                 'ㄼ','ㄽ','ㄾ','ㄿ','ㅀ','ㅁ','ㅂ','ㅄ','ㅅ','ㅆ',
                 'ㅇ','ㅈ','ㅊ','ㅋ','ㅌ','ㅍ','ㅎ']
RESERVED = ["<pad>", "<sos>", "<eos>", " "]  # <pad> == <blank>

# 자모 인덱스 테이블 만들기
ALL_JAMOS = sorted(set(CHOSUNG_LIST + JUNGSUNG_LIST + JONGSUNG_LIST))
ALL_JAMOS.remove('')  # 종성의 공백 제거

ALL_JAMOS = RESERVED + ALL_JAMOS

# 자모 사전 만들기
jamo_to_index = {j: i for i, j in enumerate(ALL_JAMOS)}
index_to_jamo = {i: j for j, i in jamo_to_index.items()}

vocab_size = len(jamo_to_index)

# 2. 자모 분해 함수
def decompose_hangul(text):
    result = ["<sos>"]
    for char in text:
        if '가' <= char <= '힣':
            code = ord(char) - ord('가')
            cho = CHOSUNG_LIST[code // (21*28)]
            jung = JUNGSUNG_LIST[(code % (21*28)) // 28]
            jong = JONGSUNG_LIST[code % 28]
            result.extend([cho, jung])
            if jong != '':
                result.append(jong)
        else:
            result.append(char)
    result.append("<eos>")
    return result

# 3. 자모 복원 함수
import re

def compose_jamos(jamo_sequence):
    # <sos>, <eos> 제거
    seq = [j for j in jamo_sequence if j not in ("<sos>", "<eos>")]
    result = ''
    i = 0
    n = len(seq)

    while i < n:
        ch = seq[i] if i < n else ''
        ju = seq[i + 1] if i + 1 < n else ''
        # 규칙: (초성, 중성) 조합이 되는 경우만 한 글자 조립 시도
        if ch in CHOSUNG_LIST and ju in JUNGSUNG_LIST:
            use_jong = False
            jong = ''

            # 후보 종성(=세 번째 자모)이 있는 경우 판단
            if i + 2 < n:
                jo = seq[i + 2]
                if jo in JONGSUNG_LIST:
                    # jo가 '다음 글자 초성'으로도 사용 가능한 자모(예: ㄱ, ㄴ, ㄷ, ㅅ, ㅇ 등)이고
                    # jo 다음이 '모음'이면 -> jo는 다음 글자의 초성으로 사용 (종성 아님)
                    if (jo in CHOSUNG_LIST) and (i + 3 < n and seq[i + 3] in JUNGSUNG_LIST):
                        use_jong = False
                    else:
                        # 그 외의 경우(문장 끝, 공백, 다음이 자음 등) -> 종성으로 사용
                        use_jong = True
                        jong = jo

            if use_jong:
                code = 0xAC00 + (
                    CHOSUNG_LIST.index(ch) * 21 * 28 +
                    JUNGSUNG_LIST.index(ju) * 28 +
                    JONGSUNG_LIST.index(jong)
                )
                result += chr(code)
                i += 3
            else:
                code = 0xAC00 + (
                    CHOSUNG_LIST.index(ch) * 21 * 28 +
                    JUNGSUNG_LIST.index(ju) * 28
                )
                result += chr(code)
                i += 2
        else:
            # 조합 불가: 있는 그대로 출력
            result += ch
            i += 1

    return result

# 4. 전체 파이프라인
def text_to_ctc_indices(text):
    text = re.sub(r'[^가-힣 ]', '', text)

    jamo_seq = decompose_hangul(text)  # 자모 분해
    jamo_seq = jamo_seq[1:] if jamo_seq[0] == " " else jamo_seq
    jamo_seq = jamo_seq[:-1] if jamo_seq[-1] == " " else jamo_seq

    indices = [jamo_to_index[j] for j in jamo_seq if j in jamo_to_index]  # 인덱싱

    compose = compose_jamos(jamo_seq)  # 복원

    return text, jamo_seq, indices, compose

# 5. 예시 실행
text = "숙주도 있어서 넣고 이것저것 해서"
text, jamo_seq, indices, compose  = text_to_ctc_indices(text)

# 결과 출력
print("✅ 원문:", text)
print("✅ 원문 분해:", jamo_seq)
print("✅ 원문 시퀀스:",indices)
print("✅ 원문 복원:", compose)

✅ 원문: 숙주도 있어서 넣고 이것저것 해서
✅ 원문 분해: ['<sos>', 'ㅅ', 'ㅜ', 'ㄱ', 'ㅈ', 'ㅜ', 'ㄷ', 'ㅗ', ' ', 'ㅇ', 'ㅣ', 'ㅆ', 'ㅇ', 'ㅓ', 'ㅅ', 'ㅓ', ' ', 'ㄴ', 'ㅓ', 'ㅎ', 'ㄱ', 'ㅗ', ' ', 'ㅇ', 'ㅣ', 'ㄱ', 'ㅓ', 'ㅅ', 'ㅈ', 'ㅓ', 'ㄱ', 'ㅓ', 'ㅅ', ' ', 'ㅎ', 'ㅐ', 'ㅅ', 'ㅓ', '<eos>']
✅ 원문 시퀀스: [1, 24, 47, 4, 27, 47, 10, 42, 3, 26, 54, 25, 26, 38, 24, 38, 3, 7, 38, 33, 4, 42, 3, 26, 54, 4, 38, 24, 27, 38, 4, 38, 24, 3, 33, 35, 24, 38, 2]
✅ 원문 복원: 숙주도 있어서 넣고 이것저것 해서


# path_info 생성

In [ ]:
# 저는 작업할때 하나의 폴더에 같은 이름으로 .wav와 .json으로 음성파일과 메타데이터 파일을 넣어놨습니다.
# ex) 노인남여_노인대화07_F_1522434093_60_경상_실내_08580.wav / 노인남여_노인대화07_F_1522434093_60_경상_실내_08580.json

In [ ]:
# 음성파일과 메타데이터 파일 불러오기

json_list = glob.glob('/data_folder/*.json')
json_list.sort()

wav_list = glob.glob('/data_folder/*.wav')
wav_list.sort()

# wav, json 정렬 확인
for json_file, wav in zip(json_list, wav_list):
  if json_file.split('.json')[0] != wav.split('.wav')[0]:
    print(json_file, wav)

In [ ]:
# 음성파일 경로와 음성의 전사 반환하는 함수
def load_one(file_name):
    with open(file_name, 'rb') as f:
        data = orjson.loads(f.read())
    return str(Path(file_name).with_suffix('.wav')), data["발화정보"]["stt"], data["녹음자정보"]["recorderId"]  # with_suffix: 확장자를 변경해주는 메소드 / .json을 .wav로 변경

# 병렬처리로 작업
with ThreadPoolExecutor(max_workers=16) as ex:
    results = list(tqdm(ex.map(load_one, json_list), total=len(json_list)))

100%|██████████| 141939/141939 [2:11:30<00:00, 17.99it/s]


In [ ]:
# 음성파일 경로와 전사된 문장 DataFrame으로 저장 후 재사용 위해 csv로 저장
path_info = pd.DataFrame(results, columns=["path", "transcript", "recorderId"])

# script안에 한글을 제외한 내용 제거
path_info["transcript_clean"] = (
    path_info["transcript"]
        .astype(str)
        .str.replace(r"[^가-힣ㄱ-ㅎㅏ-ㅣ ]", "", regex=True)
)

# recorderId 중복문제 해소 (recorderId+순서+지역)
path_info['recorderId'] = path_info['recorderId']+'_'+path_info['path'].str.split('_').str[4]+'_'+path_info['path'].str.split('_').str[5]

path_info.to_csv('path_info.csv', index=False)

In [ ]:
wav_list= path_info['path'].to_list()

In [ ]:
# 음성데이터 경로, 샘플링레이트, 멜변환차원수 변수로 넣어 멜변환해주는 함수
def audio_preprocess(wav, sr=16000, n_mels=128):
  ori_y1, sr1 = librosa.load(wav, sr=sr)
  mel_spec1 = librosa.feature.melspectrogram(y=ori_y1, sr=sr1, n_mels=n_mels)
  mel_db1 = librosa.power_to_db(mel_spec1, ref=np.max)
  with open(wav, 'rb') as f:
    wav_data = f.read()
  bytes_per_sample = 2
  duration = len(wav_data) / (sr * bytes_per_sample)

  # 멜변환된 데이터, 음성데이터길이(분,초), 음성데이터길이(STFT 개수)
  return mel_db1, round(duration, 3), mel_db1.shape[1]

In [ ]:
# 함수에 들어갈 변수정리
n_mels = 128  # 설정하고자 하는 멜변환 차원 수
sr = 16000  # 설정하고자 하는 샘플링레이트
sr_param = [sr] * len(wav_list)
mel_param = [n_mels] * len(wav_list)

# 병렬처리로 작업하는 함수
with ThreadPoolExecutor(max_workers=16) as ex:
    audio_pair_data = list(tqdm(ex.map(audio_preprocess, wav_list, sr_param, mel_param), total=len(wav_list)))

# 멜변환된 데이터, 음성데이터길이(분,초), 음성데이터길이(STFT 개수) DataFrame으로 저장
wav_info = pd.DataFrame(audio_pair_data, columns=['wav_data', 'duration', 'wav_length'])

100%|██████████| 141939/141939 [2:31:47<00:00, 15.59it/s]


In [ ]:
# json파일에 있는 문장 추출 후 토큰화 및 문장길이 반환하는 함수
def script_preprocess(text):
  seq = decompose_hangul(text)
  seq_2_id = [jamo_to_index[j] for j in seq if j in jamo_to_index]
  return text, seq_2_id, len(seq_2_id)

In [ ]:
# 병렬처리로 작업
with ThreadPoolExecutor(max_workers=16) as ex:
    scr_pair_data = list(tqdm(ex.map(script_preprocess, path_info['transcript_clean']), total=len(path_info['transcript_clean'])))

100%|██████████| 141939/141939 [00:00<00:00, 424369.55it/s]


In [ ]:
# 전사 문장, 토큰화된 문장, 문장길이 DataFrame으로 저장
seq_info = pd.DataFrame(scr_pair_data, columns=['script', 'seq', 'seq_length'])

In [ ]:
# 정리된 데이터들 확인
len(path_info),len(wav_info),len(seq_info)

(141939, 141939, 141939)

In [ ]:
# 10초 미만의 음성만 사용하기 위해 필터링
print(wav_info.loc[wav_info['duration']>10].index)

path_info.drop(wav_info.loc[wav_info['duration']>10].index,inplace=True)
seq_info.drop(wav_info.loc[wav_info['duration']>10].index,inplace=True)
wav_info.drop(wav_info.loc[wav_info['duration']>10].index,inplace=True)

path_info.reset_index(drop=True,inplace=True)
seq_info.reset_index(drop=True,inplace=True)
wav_info.reset_index(drop=True,inplace=True)

Index([   666,   2975,   5002,   5228,   5876,   7206,   8778,   9310,  15521,
        20200,
       ...
       130358, 130703, 131617, 131874, 136271, 136281, 136328, 136383, 136442,
       136786],
      dtype='int64', length=281)


In [ ]:
# 필터링 후 데이터 확인
len(path_info),len(wav_info),len(seq_info)

(141658, 141658, 141658)

In [54]:
path_info

,path,transcript,recorderId,transcript_clean
0,/content/drive/Othercomputers/내컴퓨터/pick/ᄂ...,나 때문에 오늘 아침부터 마음 고생이 많아요 좀 멀리서,e7222879_61_수도권,나 때문에 오늘 아침부터 마음 고생이 많아요 좀 멀리서
1,/content/drive/Othercomputers/내컴퓨터/pick/ᄂ...,오느라고 전철을 하나 놓치는 바람에 그 다음부터 계속,e7222879_61_수도권,오느라고 전철을 하나 놓치는 바람에 그 다음부터 계속
2,/content/drive/Othercomputers/내컴퓨터/pick/ᄂ...,늦어져 가지고 정말 너무 많이 미안하네요,e7222879_61_수도권,늦어져 가지고 정말 너무 많이 미안하네요
3,/content/drive/Othercomputers/내컴퓨터/pick/ᄂ...,언니가 아침부터 뛰느라고 정신 없었을 거 같아요,e7222879_61_수도권,언니가 아침부터 뛰느라고 정신 없었을 거 같아요
4,/content/drive/Othercomputers/내컴퓨터/pick/ᄂ...,불안해 가지고 그래서 이것도 약속인데 아침부터 너무 좀,e7222879_61_수도권,불안해 가지고 그래서 이것도 약속인데 아침부터 너무 좀
...,...,...,...,...
141653,/content/drive/Othercomputers/내컴퓨터/pick/ᄂ...,캠핑 카의 밧데리를 인산 철 배터리로 교체하자,1540293395_67_강원,캠핑 카의 밧데리를 인산 철 배터리로 교체하자
141654,/content/drive/Othercomputers/내컴퓨터/pick/ᄂ...,언제든지 캠핑장에 가지 않고 세워 두면서 캠핑할 수 있는 여건을 만들자,1540293395_67_강원,언제든지 캠핑장에 가지 않고 세워 두면서 캠핑할 수 있는 여건을 만들자
141655,/content/drive/Othercomputers/내컴퓨터/pick/ᄂ...,이렇게 준비해서 준비가 되는 대로 곧 여행을 떠날 생각입니다,1540293395_67_강원,이렇게 준비해서 준비가 되는 대로 곧 여행을 떠날 생각입니다
141656,/content/drive/Othercomputers/내컴퓨터/pick/ᄂ...,첫 번째는 건강 두 번째는 화목한 가정,1540293395_67_강원,첫 번째는 건강 두 번째는 화목한 가정


In [ ]:
# 음성데이터 최대길이 설정
# 기본값으로 멜변환된 10초길이의 데이터가 312라서 저는 312로 했습니다.
wav_max_len = 312

In [ ]:
# 제가 설계한 모델에서 음성데이터의 1/2로 예측이 출력되서 이렇게 설정했습니다.
seq_max_len = int(wav_max_len/2)
seq_max_len

156

In [ ]:
# 패딩 작업 전 데이터 복사
b_pad_wav = wav_info['wav_data']
b_pad_seq = seq_info['seq']

In [ ]:
# 음성데이터 패딩 함수
def wav_padding(wav, wav_max_len=312):
  pad_width = wav_max_len - wav.shape[1]  # 얼마나 채워야 하는지
  if pad_width > 0:
      # 오른쪽(열 끝)에 0을 채움: ((행 시작, 행 끝), (열 시작, 열 끝))
      padded = np.pad(wav, pad_width=((0, 0), (0, pad_width)), mode='constant', constant_values=-80)
  elif pad_width < 0:
      padded = wav[:,:wav_max_len]
  else:
      padded = wav  # 이미 가장 김
  return padded

In [ ]:
# 병렬처리로 작업하는 함수
with ThreadPoolExecutor(max_workers=16) as ex:
    audio_padded_data = list(tqdm(ex.map(wav_padding, b_pad_wav), total=len(b_pad_wav)))

100%|██████████| 141658/141658 [00:21<00:00, 6502.97it/s]


In [ ]:
# 문장데이터 패딩 함수
def scr_padding(scr, seq_max_len=156):
  pad_width = seq_max_len - len(scr)  # 얼마나 채워야 하는지
  if pad_width > 0:
      # 오른쪽(열 끝)에 0을 채움: ((행 시작, 행 끝), (열 시작, 열 끝))
      for _ in range(pad_width):
        scr.append(0)
  return scr

In [ ]:
# 병렬처리로 작업하는 함수
with ThreadPoolExecutor(max_workers=16) as ex:
    scr_padded_data = list(tqdm(ex.map(scr_padding, b_pad_seq), total=len(b_pad_seq)))

100%|██████████| 141658/141658 [00:00<00:00, 414449.39it/s]


In [ ]:
# 패딩된 데이터 변수에 numpy배열로 변수에 담기
x_padded_data = np.stack(audio_padded_data)
x_data = np.transpose(x_padded_data, (0, 2, 1))
x_data_length = wav_info['wav_length'].to_numpy()

y_data = np.stack(scr_padded_data)
y_data_length = seq_info['seq_length'].to_numpy()

In [ ]:
# shape 확인
x_data.shape, y_data.shape, x_data_length.shape, y_data_length.shape

((141658, 312, 128), (141658, 156), (141658,), (141658,))

In [ ]:
np.save('x_data_t312_m128_s156.npy', x_data)
np.save('y_data_t312_m128_s156.npy', y_data)
np.save('x_data_length_t312_m128_s156.npy', x_data_length)
np.save('y_data_length_t312_m128_s156.npy', y_data_length)

In [ ]:
# train, test, validation set 비율정해서 한번에 분리하기
def split_counts(n, test_ratio=0.15, val_ratio=0.18):
    if n <= 2:
        return n, 0, 0

    # 1) 전체에서 test 비율 먼저
    n_test = int(round(n * test_ratio))
    if n_test < 1:
        n_test = 1
    if n_test > n - 2:
        n_test = n - 2  # train+val 최소 2개

    # 2) 남은 것에서 train:val = 8:2
    remain = n - n_test
    n_val = int(round(remain * val_ratio))
    if n_val < 1:
        n_val = 1
    if n_val > remain - 1:
        n_val = remain - 1  # train 최소 1개

    n_train = n - n_test - n_val
    return n_train, n_val, n_test


# 원하는 컬럼을 기준으로 train_test_split(stratify)
def make_split_index_lists(df, group_col='transcript_clean', random_state=42):
    rng = np.random.RandomState(random_state)

    train_idx_list = []
    val_idx_list = []
    test_idx_list = []

    # transcript별로 묶어서 나누기
    for value, idx in df.groupby(group_col).groups.items():
        idx = np.array(list(idx))  # index 배열
        n = len(idx)

        n_train, n_val, n_test = split_counts(n)

        rng.shuffle(idx)  # 그룹 내부 섞기

        train_idx = idx[:n_train]
        val_idx   = idx[n_train:n_train + n_val]
        test_idx  = idx[n_train + n_val:n_train + n_val + n_test]

        train_idx_list.extend(train_idx.tolist())
        val_idx_list.extend(val_idx.tolist())
        test_idx_list.extend(test_idx.tolist())

    # 보기 좋게 정렬
    train_idx_list.sort()
    val_idx_list.sort()
    test_idx_list.sort()

    return train_idx_list, val_idx_list, test_idx_list


In [ ]:
# train, validation, test set index 추출

train_idx_list, val_idx_list, test_idx_list = make_split_index_lists(
    path_info,
    group_col='transcript_clean'
)

print(len(train_idx_list), len(val_idx_list), len(test_idx_list))

102172 23339 16147


In [ ]:
# 인덱스 저장

idx_list = train_idx_list+val_idx_list+test_idx_list
train_arg = np.array(train_idx_list)
val_arg = np.array(val_idx_list)
test_arg = np.array(test_idx_list)
idx_arg = np.array(idx_list)

np.save('train_idx_list.npy',train_arg)
np.save('val_idx_list.npy',val_arg)
np.save('test_idx_list.npy',test_arg)
np.save('idx_list.npy',idx_arg)